# DDS-Edge: train baseline on Kaggle

Before running: **Add Input** -> competition *State Farm Distracted Driver Detection*; Session options -> **Accelerator: GPU (P100)**, **Internet: On**.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Bat GPU: Session options -> Accelerator -> GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
%%bash
cd /kaggle/working
[ -d dds-edge ] || git clone https://github.com/ColorOfDreams/dds-edge.git
cd dds-edge && git pull -q
pip install -q timm onnx onnxruntime
pip install -q -e . --no-deps
git log --oneline -1

In [ ]:
# Link the (read-only) Kaggle dataset to where the code expects it: data/raw/statefarm
import os, glob
os.chdir('/kaggle/working/dds-edge')
hits = glob.glob('/kaggle/input/**/driver_imgs_list.csv', recursive=True)
assert hits, 'Chua Add Input State Farm? (khong thay driver_imgs_list.csv)'
src = os.path.dirname(hits[0])
print('dataset at', src)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
link = 'data/raw/statefarm'
if os.path.islink(link):  # stale or broken link from an earlier run
    os.unlink(link)
assert not os.path.exists(link), f'{link} is a real folder, remove it first'
os.symlink(src, link)
print(os.listdir('data/raw/statefarm'))

In [ ]:
# manifest: split by driver. Expect: groups -> train 18, val 4, test 4
!python -m dds_edge.dl.prepare_data

In [ ]:
# smoke test (1 epoch) - skip once it works
!python -m dds_edge.dl.train --epochs 1 --name smoke

In [ ]:
# baseline: MobileNetV3-Small, pretrained, aug, grouped6
!python -m dds_edge.dl.train --arch mobilenet_v3_small --grouping grouped6 --name mnv3_pre_aug

In [ ]:
%cd /kaggle/working/dds-edge
# other models, same protocol as the baseline. Uncomment ONE line per run.
# !python -m dds_edge.dl.train --arch resnet18 --grouping grouped6 --name resnet18_pre_aug
# !python -m dds_edge.dl.train --arch efficientnet_lite0 --grouping grouped6 --name effl0_pre_aug
# !python -m dds_edge.dl.train --arch resnet50 --grouping grouped6 --name resnet50_pre_aug  # heavy reference

In [ ]:
# package results: download runs.zip from the Output tab (/kaggle/working)
!cd /kaggle/working/dds-edge && zip -qr /kaggle/working/runs.zip runs data/processed/manifest.csv
!ls -lh /kaggle/working/runs.zip